# 📅 Day 16 — Mock Interview Practice

**Week 3 · Day 2 · Level: ADVANCED (deepening into HARD)**

Week 3 continues: **product-company level** questions — complex end-to-end scenarios, optimization trade-offs, and system-design aspects. 18 questions across 8 categories — 14 [ADVANCED] + 4 [HARD]. Today pushes deeper into sessionization, rate limiting, speculative execution, exactly-once semantics, watermarks, and conformed dimensions. Every answer should come with a trade-off articulated out loud.

| # | Category | Questions | Level |
| --- | --- | --- | --- |
| 1 | SQL Advanced | Q1-Q2 | ADVANCED / HARD |
| 2 | Python — Loop/Function/List/Dict | Q3-Q4 | ADVANCED |
| 3 | PySpark Coding | Q5-Q6 | ADVANCED |
| 4 | Spark Optimization & Deep Spark | Q7-Q8 | ADVANCED |
| 5 | ETL — Failures, Late Data, Schema Handling | Q9-Q11 | ADVANCED / HARD |
| 6 | Data Modeling | Q12-Q13 | ADVANCED |
| 7 | Snowflake | Q14-Q15 | ADVANCED |
| 8 | Monitoring, Logging & Orchestration | Q16-Q18 | ADVANCED / HARD |

**Difficulty progression**: Day 15 introduced Week 3 with z-score bands, range joins, caching layers, and bitemporal facts. Day 16 deepens with cohort retention grids, sessionization via gaps-and-islands, skewed-data salting, SCD Type 6 hybrids, exactly-once end-to-end, and circuit breakers — each a step more complex than its Day 15 counterpart.

## Q1 [ADVANCED] SQL Advanced — Cohort Retention Grid from Activity Events

**Scenario**: A product analytics team needs a **cohort retention grid**: group users by their signup month (the cohort), then for each subsequent month compute what % of the cohort was active. This is the foundational query behind every retention dashboard — and interviewers love it because it requires multi-level aggregation with conditional logic.

**Requirements**:
1. Build the cohort retention grid: rows = cohort month, columns = months since signup (0, 1, 2, …), values = % of cohort active in that month (cohort_month_size = users active in month 0).
2. Month 0 retention is always 100% (every user is active in their signup month by definition).
3. Handle cohorts of different sizes — a 3-user cohort with 2 active in month 1 = 66.7%, not the same as a 100-user cohort with 2 active.
4. Show the grid as a pivot with explicit month-since-signup columns.

**Sample data**: 7 users across 3 signup cohorts (Jan, Feb, Mar 2026), with activity spanning 4 months. Expected: Jan cohort (3 users) shows 100% / 67% / 33% / 33% for months 0-3.

In [0]:
# ── Solution Q1: cohort retention grid via conditional aggregation + pivot ─────────
from pyspark.sql.functions import col, min as spark_min, countDistinct, when, lit
from pyspark.sql.types import *

# Sample: (user_id, signup_month, active_month)
activity = [
    (1, "2026-01", "2026-01"), (1, "2026-01", "2026-02"), (1, "2026-01", "2026-04"),
    (2, "2026-01", "2026-01"), (2, "2026-01", "2026-02"),
    (3, "2026-01", "2026-01"), (3, "2026-01", "2026-03"),
    (4, "2026-02", "2026-02"), (4, "2026-02", "2026-03"),
    (5, "2026-02", "2026-02"),
    (6, "2026-03", "2026-03"), (6, "2026-03", "2026-04"),
    (7, "2026-03", "2026-03"),
]

raw = spark.createDataFrame(activity, ["user_id", "signup_month", "active_month"])

# Step 1: cohort size = distinct users per signup month (month-0 denominator)
cohort_sizes = (raw.groupBy("signup_month")
                  .agg(countDistinct("user_id").alias("cohort_size")))

# Step 2: months_since_signup = months between active and signup
with_gap = (raw.withColumn("months_since",
             (col("active_month").substr(1, 4).cast("int") * 12 + col("active_month").substr(6, 2).cast("int"))
             - (col("signup_month").substr(1, 4).cast("int") * 12 + col("signup_month").substr(6, 2).cast("int"))
            ))

# Step 3: distinct active users per (cohort, months_since)
retention = (with_gap.groupBy("signup_month", "months_since")
               .agg(countDistinct("user_id").alias("active_users")))

# Step 4: join cohort sizes and compute retention rate
retention_pct = (retention.join(cohort_sizes, "signup_month")
                   .withColumn("retention_pct", (col("active_users") / col("cohort_size") * 100).cast("decimal(5,1)")))

# Step 5: pivot into grid
grid = (retention_pct.groupBy("signup_month")
         .pivot("months_since", [0, 1, 2, 3])
         .agg(spark_min("retention_pct"))
         .orderBy("signup_month"))

grid.show()

# Verify: Jan cohort (3 users) month 0 = 100%, month 1 = 2/3 = 66.7%, month 2 = 1/3 = 33.3%
jan = grid.filter("signup_month = '2026-01'").first()
assert jan["0"] == 100.0, f"Month 0 should be 100%, got {jan['0']}"
assert jan["1"] == 66.7, f"Month 1 should be 66.7%, got {jan['1']}"
assert jan["2"] == 33.3, f"Month 2 should be 33.3%, got {jan['2']}"
print("Cohort retention grid verified ✓")
print("\nInterview insight: the key is the DENOMINATOR — cohort_size is fixed at month 0.")
print("A common mistake is dividing by current-month active count instead of original cohort size.")

## Q2 [HARD] SQL Advanced — Sessionization via Gaps-and-Islands

**Scenario**: You have a clickstream of user events with timestamps. A **session** is defined as a sequence of events by the same user where no two consecutive events are more than 30 minutes apart. If a user clicks at 10:00, 10:20, 11:00, that's two sessions: [10:00, 10:20] and [11:00] (40-min gap between 10:20 and 11:00). This is a classic gaps-and-islands problem asked at Meta, Google, and Amazon.

**Requirements**:
1. For each user, identify session boundaries using the 30-minute inactivity gap rule.
2. Assign a `session_id` to each event — events in the same session share the same id.
3. Produce a per-session summary: user_id, session_start, session_end, event_count, session_duration_minutes.
4. Handle edge cases: a single event is a session (duration 0), events exactly 30 minutes apart are in the SAME session (≤ 30 min = same session).

**Sample data**: user 1 has events at 10:00, 10:20, 11:00 (gap = 40 min → new session), 11:05, 12:00 (gap = 55 min → new session). Expected: 3 sessions for user 1.

In [0]:
# ── Solution Q2: sessionization via gaps-and-islands with window functions ──────
from pyspark.sql import Window
from pyspark.sql.functions import col, lag, when, sum as spark_sum, min as spark_min, max as spark_max, count, unix_timestamp, expr

# Sample clickstream: (event_id, user_id, event_ts)
events = [
    (1, 1, "2026-01-01 10:00:00"),
    (2, 1, "2026-01-01 10:20:00"),
    (3, 1, "2026-01-01 11:00:00"),   # 40-min gap → new session
    (4, 1, "2026-01-01 11:05:00"),
    (5, 1, "2026-01-01 12:00:00"),   # 55-min gap → new session
    (6, 2, "2026-01-01 09:00:00"),
    (7, 2, "2026-01-01 09:25:00"),   # 25-min gap → same session
    (8, 2, "2026-01-01 10:00:00"),   # 35-min gap → new session
]

df = spark.createDataFrame(events, ["event_id", "user_id", "event_ts"])
df = df.withColumn("event_ts", col("event_ts").cast("timestamp"))

# Step 1: lag to get previous event timestamp per user
w = Window.partitionBy("user_id").orderBy("event_ts")
df = df.withColumn("prev_ts", lag("event_ts").over(w))

# Step 2: gap in minutes; flag new session when gap > 30 or no previous (first event)
df = (df.withColumn("gap_minutes",
        when(col("prev_ts").isNotNull(),
             (unix_timestamp("event_ts") - unix_timestamp("prev_ts")) / 60)
        .otherwise(0))
      .withColumn("is_new_session", when(col("gap_minutes") > 30, 1).otherwise(0)))

# Step 3: cumulative sum of is_new_session → session number within user
w2 = Window.partitionBy("user_id").orderBy("event_ts").rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)
df = df.withColumn("session_num", spark_sum("is_new_session").over(w2))

# Step 4: global session_id (user_id + session_num for uniqueness)
df = df.withColumn("session_id", expr("concat(user_id, '-', session_num + 1)"))

# Step 5: per-session summary
session_summary = (df.groupBy("user_id", "session_id")
                     .agg(spark_min("event_ts").alias("session_start"),
                          spark_max("event_ts").alias("session_end"),
                          count("event_id").alias("event_count"))
                     .withColumn("duration_minutes",
                        (unix_timestamp("session_end") - unix_timestamp("session_start")) / 60)
                     .orderBy("user_id", "session_start"))

session_summary.show(truncate=False)

# Verify: user 1 has 3 sessions, user 2 has 2 sessions
assert session_summary.filter("user_id = 1").count() == 3, "User 1 should have 3 sessions"
assert session_summary.filter("user_id = 2").count() == 2, "User 2 should have 2 sessions"
print("Sessionization verified ✓ — 3 sessions for user 1, 2 sessions for user 2")
print("\nGaps-and-islands pattern: lag → gap → flag → cumulative sum → group.")
print("The cumulative sum of 'new session' flags creates a monotonically increasing session number.")

## Q3 [ADVANCED] Python — LRU Cache Implementation (Doubly-Linked List + Dict)

**Scenario**: You're building a caching layer for a recommendation service that fetches user profiles from an external API. The cache must evict the **least recently used** entry when capacity is exceeded. Implement it from scratch — no `functools.lru_cache` allowed.

**Requirements**:
1. Implement `LRUCache(capacity)` with `get(key)` and `put(key, value)` — both O(1).
2. Use a **doubly-linked list** for O(1) recency updates + a **dict** for O(1) key lookup.
3. `get` moves the accessed node to the head (most recently used). Returns -1 if key not found.
4. `put` inserts/updates at head. If capacity exceeded, evicts the tail (LRU).
5. Include test cases: basic ops, eviction order, capacity-1 edge case, update existing key.

**Expected**: cache(2); put(1,'a'), put(2,'b'), get(1), put(3,'c'), get(2) → -1 (2 was LRU after 1 was accessed).

In [0]:
# ── Solution Q3: LRU cache with doubly-linked list + dict ────────────────────

class DLinkedNode:
    __slots__ = ('key', 'value', 'prev', 'next')  # lean nodes for memory efficiency
    def __init__(self, key=0, value=0):
        self.key = key
        self.value = value
        self.prev = None
        self.next = None

class LRUCache:
    def __init__(self, capacity: int):
        self.capacity = capacity
        self.cache = {}  # key -> DLinkedNode
        # Sentinel head/tail to avoid null checks on insert/remove
        self.head = DLinkedNode()
        self.tail = DLinkedNode()
        self.head.next = self.tail
        self.tail.prev = self.head

    def _remove(self, node):
        """Unlink a node from the list."""
        node.prev.next = node.next
        node.next.prev = node.prev

    def _add_to_head(self, node):
        """Insert right after the head sentinel (most recently used)."""
        node.prev = self.head
        node.next = self.head.next
        self.head.next.prev = node
        self.head.next = node

    def _move_to_head(self, node):
        self._remove(node)
        self._add_to_head(node)

    def _pop_tail(self):
        """Evict and return the LRU node (right before tail sentinel)."""
        lru = self.tail.prev
        self._remove(lru)
        return lru

    def get(self, key):
        if key not in self.cache:
            return -1
        self._move_to_head(self.cache[key])  # mark as most recently used
        return self.cache[key].value

    def put(self, key, value):
        if key in self.cache:
            self.cache[key].value = value
            self._move_to_head(self.cache[key])
        else:
            node = DLinkedNode(key, value)
            self.cache[key] = node
            self._add_to_head(node)
            if len(self.cache) > self.capacity:
                evicted = self._pop_tail()
                del self.cache[evicted.key]

# ── Test cases ────────────────────────────────────────────────────────────────
c = LRUCache(2)
c.put(1, 'a')
c.put(2, 'b')
assert c.get(1) == 'a'         # access 1 → 1 is now MRU, 2 is LRU
c.put(3, 'c')                  # evicts 2 (the LRU)
assert c.get(2) == -1          # 2 was evicted
assert c.get(3) == 'c'
assert c.get(1) == 'a'         # 1 still there
c.put(4, 'd')                  # evicts 3 (now LRU after 1 was accessed)
assert c.get(3) == -1
assert c.get(4) == 'd'
assert c.get(1) == 'a'
print("All LRU cache tests passed ✓")

# Edge case: capacity = 1
c1 = LRUCache(1)
c1.put(1, 'x')
c1.put(2, 'y')
assert c1.get(1) == -1
assert c1.get(2) == 'y'
print("Capacity-1 edge case passed ✓")

# Edge case: update existing key doesn't evict
c2 = LRUCache(2)
c2.put(1, 'a')
c2.put(2, 'b')
c2.put(1, 'updated')           # update, not insert → no eviction
assert c2.get(1) == 'updated'
assert c2.get(2) == 'b'
print("Update-existing-key edge case passed ✓")

## Q4 [ADVANCED] Python — Rate Limiter: Token Bucket + Sliding Window Log

**Scenario**: You're building an API gateway that needs to limit requests per user. The product team asks for two algorithms: a **token bucket** (allows bursts up to capacity, refills continuously) and a **sliding window log** (strict max requests per time window, no bursts beyond the limit). Both are core system design primitives asked at Amazon, Stripe, and Uber.

**Requirements**:
1. Implement `TokenBucketRateLimiter(capacity, rate)`: `allow()` returns True if a token is available, refills at `rate` tokens/sec up to `capacity`.
2. Implement `SlidingWindowLogLimiter(max_req, window_sec)`: `allow(now)` returns True if fewer than `max_req` requests have been made in the past `window_sec` seconds.
3. Explain the trade-offs: token bucket allows bursts (good for human users), sliding window is strict (good for billing APIs).
4. Discuss distributed rate limiting: how would you shard the limiter state across multiple API gateway instances?

**Expected**: Token bucket with capacity=5, rate=1/sec → first 5 calls pass instantly, 6th fails, after 1 sec 1 more passes. Sliding window with max=3, window=1s → 3 pass, 4th fails, after 1s window resets.

In [0]:
# ── Solution Q4: rate limiter — token bucket + sliding window log ──────────
import time
from collections import deque

class TokenBucketRateLimiter:
    """Token bucket: refills at `rate` tokens/sec, max `capacity` tokens."""
    def __init__(self, capacity, rate):
        self.capacity = capacity
        self.rate = rate
        self.tokens = capacity
        self.last_refill = time.monotonic()

    def _refill(self):
        now = time.monotonic()
        elapsed = now - self.last_refill
        self.tokens = min(self.capacity, self.tokens + elapsed * self.rate)
        self.last_refill = now

    def allow(self):
        self._refill()
        if self.tokens >= 1:
            self.tokens -= 1
            return True
        return False

class SlidingWindowLogLimiter:
    """Sliding window log: max `max_req` requests in `window_sec`."""
    def __init__(self, max_req, window_sec):
        self.max_req = max_req
        self.window_sec = window_sec
        self.hits = deque()

    def allow(self, now=None):
        if now is None:
            now = time.monotonic()
        while self.hits and now - self.hits[0] >= self.window_sec:
            self.hits.popleft()
        if len(self.hits) < self.max_req:
            self.hits.append(now)
            return True
        return False

# ── Test: Token Bucket ──
tb = TokenBucketRateLimiter(5, 1.0)
results = [tb.allow() for _ in range(5)]
assert all(results), f"First 5 should pass: {results}"
assert not tb.allow(), "6th should fail (no refill yet)"
time.sleep(1.0)
assert tb.allow(), "After 1 sec, 1 token refilled"
print("Token bucket: 5 instant + 6th denied + 1 after 1s → PASS")

# ── Test: Sliding Window Log ──
sw = SlidingWindowLogLimiter(3, 1.0)
assert sw.allow(now=0.0)
assert sw.allow(now=0.3)
assert sw.allow(now=0.6)
assert not sw.allow(now=0.9), "4th in same window should fail"
assert sw.allow(now=1.1), "New window should allow"
print("Sliding window: 3 per 1s, 4th denied, reset after 1s → PASS")

print("\n── Trade-offs ──")
print("  Token bucket: allows bursts up to capacity → good for human users")
print("  (a user who hasn't made requests in a while gets a burst budget).")
print("  Sliding window log: strict max per window → good for billing APIs")
print("  (never exceeds N requests per minute, period).")
print()
print("── Distributed rate limiting ──")
print("  Single-instance: state in memory (fast, but lost on restart).")
print("  Multi-instance: store counters in Redis (INCR + EXPIRE for sliding window,")
print("  or Lua script for token bucket). Partition by user_id to shard load.")
print("  Trade-off: Redis round-trip adds ~0.5ms latency per request.")
print("  Alternative: local limiter + periodic sync to Redis (approximate, allows")
print("  brief over-limit during sync intervals — acceptable for most products).")
print("\nInterview one-liner: 'Token bucket for burst tolerance, sliding window for")
print("strict limits — in distributed mode, Redis with Lua scripts is the standard.'")

## Q5 [ADVANCED] PySpark Coding — Handling Skewed Data with Salting

**Scenario**: You have a `facts` table where 80% of rows have `category = 'ELECTRONICS'` and 20% are spread across 99 other categories. A join with `dim_categories` on `category` causes one task to process the entire ELECTRONICS partition while the other 99 tasks finish in seconds. The classic **data skew** problem.

**Requirements**:
1. Implement the **salting technique**: append a random salt suffix (0..N-1) to the skewed key on the large table, and explode the small table N times with matching salts.
2. After the join, strip the salt to recover original rows.
3. Show the before (skewed) and after (salted) partition distribution.
4. Explain when salting wins vs broadcast join vs AQE skew splitting (Spark 3+/serverless).

**Sample data**: 100 facts rows (85 with 'ELECTRONICS', 15 spread across others). dim has 100 rows (1 per category). Salt factor = 5.

In [0]:
# ── Solution Q5: salting technique for skewed joins ────────────────────────
from pyspark.sql.functions import col, concat, lit, rand, floor, explode, array, broadcast
from pyspark.sql.types import *
import random

# Build skewed facts: 85 'ELECTRONICS', 5 each for 'BOOKS', 'CLOTHING', 'FOOD', 'TOYS'
fact_rows = [(i, "ELECTRONICS") for i in range(85)]
fact_rows += [(i, c) for c in ["BOOKS", "CLOTHING", "FOOD", "TOYS"] for i in range(85, 100)]
facts = spark.createDataFrame(fact_rows, ["id", "category"])

dim_rows = [(c, c.lower() + "_dept") for c in
            ["ELECTRONICS", "BOOKS", "CLOTHING", "FOOD", "TOYS"]]
dim = spark.createDataFrame(dim_rows, ["category", "dept_name"])

# ── BEFORE: naive join → all ELECTRONICS rows land in ONE partition ──
naive = facts.join(dim, "category")
print(f"Naive join row count: {naive.count()}")

# ── AFTER: salting technique ──
SALT_N = 5

# 1. Add salt to the LARGE table (facts): random 0..SALT_N-1 appended to key
facts_salted = (facts.withColumn("salt", floor(rand() * SALT_N).cast("int"))
                     .withColumn("salted_key", concat(col("category"), lit("_"), col("salt"))))

# 2. Explode the SMALL table (dim) SALT_N times: one row per salt value
dim_exploded = (dim.withColumn("salt_id", explode(array([lit(i) for i in range(SALT_N)])))
                  .withColumn("salted_key", concat(col("category"), lit("_"), col("salt_id"))))

# 3. Join on the salted key — the 85 ELECTRONICS rows are now spread across 5 partitions
salted_join = facts_salted.join(dim_exploded, "salted_key")

# 4. Strip salt and verify correctness
result = salted_join.select(col("id"), col("category"), col("dept_name")).dropDuplicates(["id", "category"])

result_count = result.count()
assert result_count == 100, f"Expected 100 rows after salting, got {result_count}"
assert result.filter("category = 'ELECTRONICS'").count() == 85
print(f"Salted join row count: {result_count} (100 ✓)")
print(f"ELECTRONICS rows: {result.filter(\"category = 'ELECTRONICS'\").count()} (85 ✓)")

print("\nTalk track — when to use which technique:")
print("  SALTING: manual, works on all Spark versions, best when you KNOW the skewed keys.")
print("  BROADCAST JOIN: if the dim is small enough, broadcast it → no shuffle at all.")
print("  AQE SKEW SPLITTING (Spark 3+/serverless): spark.sql.adaptive.skewJoin.enabled=true")
print("    automatically splits fat partitions — the modern default, but you still need to")
print("    understand salting for interviews and legacy clusters.")
print("  Rule of thumb: try AQE first → if still skewed → broadcast if dim fits → salt as last resort.")

## Q6 [ADVANCED] PySpark Coding — SCD Type 2 MERGE INTO with Delta Lake

**Scenario**: A `dim_customer` table needs to track historical changes to customer attributes. When a customer's `tier` changes from 'STANDARD' to 'GOLD', the old row is expired (`is_current = false`, `end_date = today`) and a new row is inserted with `is_current = true`. This is **SCD Type 2** — essential for point-in-time analysis ("what tier was this customer in last quarter?").

**Requirements**:
1. Create the SCD2 Delta table with: `customer_sk`, `customer_id`, `tier`, `email`, `start_date`, `end_date`, `is_current`.
2. Implement the MERGE INTO logic: on match (same tier) → update; on match (tier changed) → expire old + insert new; on no match → insert new.
3. Show that after two updates, a customer has 3 rows: original (expired), first change (expired), current (active).
4. Demonstrate point-in-time query: "what tier was customer 1 on 2026-01-15?"

**Sample data**: Customer 1 starts as STANDARD on 2026-01-01, changes to GOLD on 2026-01-15, changes to VIP on 2026-02-01.

In [0]:
# ── Solution Q6: SCD Type 2 with Delta MERGE INTO ──────────────────────────────
from pyspark.sql.functions import col, lit, current_date, to_date, when

# Initialize the SCD2 table with the initial load
TARGET = "workspace.default.d16_q6_dim_customer"
spark.sql(f"DROP TABLE IF EXISTS {TARGET}")

initial = spark.createDataFrame([
    (1, "CUST-001", "STANDARD", "cust1@email.com"),
    (2, "CUST-002", "STANDARD", "cust2@email.com"),
], ["customer_sk", "customer_id", "tier", "email"])

initial = (initial.withColumn("start_date", to_date(lit("2026-01-01")))
                .withColumn("end_date", to_date(lit("9999-12-31")))
                .withColumn("is_current", lit(True)))

initial.write.format("delta").saveAsTable(TARGET)

# ── Function: apply SCD2 update from a batch of source changes ──
def apply_scd2(source_df, target_table, update_date_str):
    """MERGE source changes into the SCD2 target: expire old, insert new."""
    source_df = source_df.withColumn("new_start_date", to_date(lit(update_date_str)))
    source_df.createOrReplaceTempView("scd2_source")

    spark.sql(f"""
    -- Step 1: Expire current rows whose tier changed
    MERGE INTO {target_table} AS t
    USING scd2_source AS s
      ON t.customer_id = s.customer_id AND t.is_current = true
    WHEN MATCHED AND t.tier != s.tier
      THEN UPDATE SET
        t.end_date = s.new_start_date,
        t.is_current = false
    """)

    # Step 2: Insert new current rows for changed attributes
    new_rows = spark.sql(f"""
    SELECT s.customer_id, s.tier, s.email,
           s.new_start_date AS start_date,
           to_date('9999-12-31') AS end_date,
           true AS is_current
    FROM scd2_source s
    LEFT JOIN {target_table} t
      ON t.customer_id = s.customer_id AND t.is_current = true
    WHERE t.customer_id IS NULL
    """)
    new_rows.write.format("delta").mode("append").saveAsTable(target_table)

# ── Update 1: customer 1 → GOLD on 2026-01-15 ──
batch1 = spark.createDataFrame([(1, "CUST-001", "GOLD", "cust1@email.com")],
                               ["customer_id", "customer_id_str", "tier", "email"])
# fix: use proper column names
batch1 = spark.createDataFrame([("CUST-001", "GOLD", "cust1@email.com")],
                               ["customer_id", "tier", "email"])
apply_scd2(batch1, TARGET, "2026-01-15")

# ── Update 2: customer 1 → VIP on 2026-02-01 ──
batch2 = spark.createDataFrame([("CUST-001", "VIP", "cust1@email.com")],
                               ["customer_id", "tier", "email"])
apply_scd2(batch2, TARGET, "2026-02-01")

# ── Verify: customer 1 should have 3 rows, 1 current ──
hist = spark.sql(f"SELECT customer_id, tier, start_date, end_date, is_current FROM {TARGET} WHERE customer_id = 'CUST-001' ORDER BY start_date")
hist.show(truncate=False)

row_count = hist.count()
current_count = hist.filter("is_current = true").count()
assert row_count == 3, f"Expected 3 rows, got {row_count}"
assert current_count == 1, f"Expected 1 current row, got {current_count}"
assert hist.filter("is_current = true").first()["tier"] == "VIP"
print("SCD Type 2 verified ✓ — 3 rows (STANDARD→GOLD→VIP), 1 current")

# ── Point-in-time query: what tier was customer 1 on 2026-01-15? ──
pit = spark.sql(f"""
SELECT tier FROM {TARGET}
WHERE customer_id = 'CUST-001'
  AND start_date <= '2026-01-15' AND end_date > '2026-01-15'
""").collect()[0]["tier"]
assert pit == "GOLD", f"On 2026-01-15, tier should be GOLD, got {pit}"
print(f"Point-in-time query: customer 1 on 2026-01-15 → tier = {pit} ✓")

## Q7 [ADVANCED] Spark Optimization — Adaptive Query Execution (AQE) Internals

**Scenario**: You inherited a pipeline where someone hard-coded `spark.sql.shuffle.partitions = 200`. The team manually tunes partitions for every join. On Spark 3.0+, AQE was enabled and things "just worked better" — but nobody can explain WHY. Your interviewer asks: "Walk me through the three AQE optimizations and when each fires."

**Requirements**:
1. Explain the **three AQE optimizations** in detail: (a) dynamic partition coalescing, (b) dynamic join strategy switch, (c) dynamic skew splitting.
2. For each: what problem it solves, what triggers it, and what config controls it.
3. Explain the **runtime statistics** that AQE collects between stages and why it can only optimize after the first stage completes (not mid-stage).
4. Describe what happens on Databricks serverless (auto-managed) vs classic shared clusters.

**Expected output**: a structured explanation with config knobs and the talk-track for an interview.

In [0]:
# ── Solution Q7: AQE internals — the three optimizations ──────────────────────
print("AQE (Adaptive Query Execution) — Three Optimizations")
print("=" * 70)
print()
print("1. DYNAMIC PARTITION COALESCING")
print("   Problem: spark.sql.shuffle.partitions defaults to 200. A 200MB post-shuffle")
print("   dataset → 200 partitions of ~1MB each → tiny tasks, scheduling overhead dominates.")
print("   Fix: AQE coalesces small adjacent partitions into bigger ones based on actual")
print("   post-shuffle data size. Target: spark.sql.adaptive.advisoryPartitionSizeInBytes")
print("   (default 64MB). Coalesces 200 tiny → ~3-4 well-sized partitions.")
print("   Config: spark.sql.adaptive.coalescePartitions.enabled (default true)")
print("           spark.sql.adaptive.advisoryPartitionSizeInBytes = 64MB")
print()
print("2. DYNAMIC JOIN STRATEGY SWITCH")
print("   Problem: At plan time, Spark estimates a table at 10GB → chooses sort-merge join.")
print("   After the map stage runs, actual size = 50MB → broadcast would have been cheaper.")
print("   Fix: AQE re-evaluates join strategy using ACTUAL statistics from completed stages.")
print("   If a side is now under spark.sql.adaptive.autoBroadcastJoinThreshold (default 10MB,"),
print("   or 0 = use the static threshold), it switches to broadcast.")
print("   Config: spark.sql.adaptive.localShuffleReader.enabled (enables the broadcast switch)")
print()
print("3. DYNAMIC SKEW SPLITTING")
print("   Problem: One partition is 10x larger than others → the straggler task dominates runtime.")
print("   Fix: AQE detects skewed partitions (size > median × spark.sql.adaptive.skewJoin.factor,")
print("   default 5, AND above spark.sql.adaptive.skewJoin.skewedPartitionFactor threshold) and")
print("   splits them into multiple smaller partitions that are processed in parallel.")
print("   Config: spark.sql.adaptive.skewJoin.enabled = true")
print("           spark.sql.adaptive.skewJoin.skewedPartitionFactor = 5")
print()
print("WHY AQE FIRES BETWEEN STAGES, NOT MID-STAGE:")
print("   AQE collects statistics (partition sizes, row counts) from the COMPLETED map stage.")
print("   It can only re-optimize the NEXT stage's plan, not the one already running.")
print("   This is why AQE helps for pipelines with multiple stages — the first stage still runs")
print("   with the static plan, but subsequent stages benefit from runtime stats.")
print()
print("DATABRICKS SERVERLESS vs CLASSIC:")
print("   Serverless: AQE is ON by default with 'auto' partitioning — you don't set")
print("   spark.sql.shuffle.partitions at all; the platform manages it entirely.")
print("   Classic shared: set spark.sql.adaptive.enabled = true explicitly (default true in 3.2+).")

# Show current AQE settings
print("\n── Current AQE settings on this cluster ──")
for c in [
    "spark.sql.adaptive.enabled",
    "spark.sql.adaptive.coalescePartitions.enabled",
    "spark.sql.adaptive.skewJoin.enabled",
    "spark.sql.adaptive.advisoryPartitionSizeInBytes",
]:
    try:
        print(f"  {c} = {spark.conf.get(c)}")
    except Exception:
        print(f"  {c} = (not set / not available)")
print("\nInterview one-liner: 'AQE turns Spark from a compile-time optimizer into a")
print("runtime optimizer — it re-plans between stages using actual statistics.'")

## Q8 [ADVANCED] Spark Optimization — Speculative Execution: Detecting & Handling Straggler Tasks

**Scenario**: A Spark job with 1000 tasks finishes 998 tasks in 5 minutes, but task #847 and #823 take 25 minutes each. The stage doesn't complete until the last task finishes. The Spark UI shows no errors — just two tasks with 5x the median duration. This is the classic **straggler** problem.

**Requirements**:
1. Explain what **speculative execution** is: Spark launches duplicate copies of slow tasks, and takes the first result, killing the other.
2. How Spark detects stragglers: the `speculativeMultiplier` (default 1.5x) and `speculativeQuantile` (default 0.75) thresholds.
3. Enable/configure it: `spark.speculation`, `spark.speculation.multiplier`, `spark.speculation.quantile`, and why it's OFF by default on most clusters.
4. When speculative execution helps (bad node, GC spike, HDFS slow read) vs when it HURTS (idempotent sinks, non-deterministic UDFs, shared resource contention).
5. The modern alternative: AQE skew splitting handles the root cause (uneven data) vs speculation treats the symptom (slow task).

**Expected output**: config examples + the decision talk-track.

In [0]:
# ── Solution Q8: speculative execution — straggler detection & mitigation ──────
print("SPECULATIVE EXECUTION — HOW IT WORKS:")
print("  Spark monitors task durations within a stage. When a task takes longer than")
print("  (median_duration × speculativeMultiplier) AND that task is in the slowest")
print("  (1 - speculativeQuantile) percentile, Spark launches a SPECULATIVE COPY")
print("  on a different executor. Whichever finishes first wins; the other is killed.")
print()
print("CONFIG KNOBS:")
print("  spark.speculation = true                    # OFF by default on most clusters")
print("  spark.speculation.multiplier = 1.5          # task must be 1.5x median to qualify")
print("  spark.speculation.quantile = 0.75           # only check the slowest 25% of tasks")
print("  spark.speculation.minTaskTime = 100ms       # don't speculate tiny tasks")
print("  spark.locality.wait = 3s                    # speculation respects locality levels")
print()
print("WHY IT'S OFF BY DEFAULT:")
print("  Speculation DOUBLES work for slow tasks — if the cluster is full, the speculative")
print("  copy preempts other jobs' tasks. On a shared cluster, this causes cascading delays.")
print("  It also requires idempotent sinks (writing twice must be safe) and deterministic")
print("  processing (same input → same output). Many pipelines violate this.")
print()
print("WHEN SPECULATION HELPS:")
print("  ✓ Bad node: one executor has failing disks → slow reads. Speculation runs on a")
print("    healthy node and finishes 10x faster.")
print("  ✓ GC spike: one task hits a full-GC pause. Speculation likely avoids it on another JVM.")
print("  ✓ HDFS slow read: one block replica is on a busy datanode. Speculation reads a faster replica.")
print()
print("WHEN SPECULATION HURTS:")
print("  ✗ Non-idempotent sinks: writing to a key-value store with INSERT (not MERGE)")
print("    → speculative copy creates duplicate rows.")
print("  ✗ Non-deterministic UDFs: random sampling, CURRENT_TIMESTAMP() → different results")
print("    from the two copies; Spark picks one, the other is wasted compute.")
print("  ✗ Resource contention: on a full cluster, the speculative task steals an executor")
print("    slot from another job, slowing the whole pipeline.")
print()
print("SPECULATION vs AQE SKEW SPLITTING:")
print("  Speculation treats the SYMPTOM (slow task) by adding more compute.")
print("  AQE skew splitting treats the CAUSE (uneven data) by splitting the fat partition.")
print("  Rule: if the straggler is caused by data skew → fix with AQE/salting (Q5).")
print("  If it's caused by a transient infra issue (bad node, GC) → speculation is right.")
print("  In practice: enable AQE skew splitting first (it's on by default in Databricks),")
print("  add speculation only when you still see stragglers with uniform partition sizes.")
print()
print("Interview one-liner: 'Speculation is a band-aid for stragglers — use it for")
print("transient infra issues, but fix data skew with AQE, not with more compute.'")

# Show current speculation settings
print("\n── Current cluster settings ──")
for c in ["spark.speculation", "spark.speculation.multiplier", "spark.speculation.quantile"]:
    try:
        print(f"  {c} = {spark.conf.get(c)}")
    except Exception:
        print(f"  {c} = (not set)")

## Q9 [ADVANCED] ETL — Watermarks & Late Data Handling in Structured Streaming

**Scenario**: A Structured Streaming job processes real-time events from Kafka. Events arrive out of order (network delays, producer batching). You need to define a **watermark** so the system knows how long to wait for late events before finalizing a window aggregation. The choice between `append` and `update` output mode determines whether late events update previous results or are dropped.

**Requirements**:
1. Explain what a **watermark** is: `withWatermark("event_ts", "10 minutes")` — the system tracks the max event-time seen and drops events older than `max_event_time - threshold`.
2. Compare **append mode** (outputs only final, non-updatable results — drops late events) vs **update mode** (updates previous aggregations with late data — never drops, but re-writes).
3. Show the trade-off: a larger watermark threshold means more state kept (memory) and longer latency (waiting for late events); a smaller threshold drops more late events.
4. Implement a demo: events arriving out of order, show which are "on time" vs "late" relative to the watermark.
5. Name the three things that must be set for watermarked windowed aggregation: watermark, window, and output mode.

**Expected output**: watermark concept demo + the append-vs-update decision talk-track.

In [0]:
# ── Solution Q9: watermarks and late data in Structured Streaming ────────────
from pyspark.sql.functions import col, lit

# Simulate watermark concept (batch demo — actual streaming uses withWatermark API)
# Events arrive in this order (by processing time), but event_ts is out of order:
events = [
    ("e1", "2026-10-01 10:00:00", 100),  # on time
    ("e2", "2026-10-01 10:05:00", 200),  # on time
    ("e3", "2026-10-01 10:15:00", 150),  # on time (current max)
    ("e4", "2026-10-01 10:30:00", 300),  # on time (new max)
    ("e5", "2026-10-01 10:10:00", 50),   # LATE: event_ts < watermark
]

# With watermark = 10 minutes:
# After e4 arrives (max_event_time = 10:30), watermark = 10:30 - 10min = 10:20
# e5 (event_ts = 10:10) is BELOW watermark → DROPPED in append mode
threshold_min = 10
max_event_ts = "2026-10-01 10:30:00"
watermark_ts = "2026-10-01 10:20:00"

print("── Watermark concept demo ──")
print(f"  Threshold: {threshold_min} minutes")
print(f"  Max event-time seen: {max_event_ts}")
print(f"  Watermark = max - threshold = {watermark_ts}")
print()
for eid, ts, val in events:
    status = "ON TIME" if ts >= watermark_ts else "DROPPED (late)"
    print(f"  {eid}: event_ts={ts}, value={val:4d} → {status}")

print()
print("── Append mode vs Update mode ──")
print("  APPEND mode:")
print("    - Outputs a window result ONLY when the watermark passes the window end.")
print("    - Late events (below watermark) are DROPPED — the result is already emitted.")
print("    - Use case: downstream sink can't handle updates (e.g., append-only log, Kafka).")
print("    - Latency: you wait threshold minutes after the window closes before seeing results.")
print()
print("  UPDATE mode:")
print("    - Updates previous aggregation results as late events arrive.")
print("    - Late events are NOT dropped (until state TTL expires) — they update the count.")
print("    - Use case: downstream sink supports updates (Delta Lake MERGE, memory sink).")
print("    - Latency: results appear immediately, but may change as late data arrives.")
print()
print("── The watermark threshold trade-off ──")
print("  Small threshold (e.g., 1 min): low latency, low memory, BUT drops more late events.")
print("  Large threshold (e.g., 60 min): catches most late events, BUT high state cost")
print("    (all open windows stay in state for 60 min) and high latency (wait 60 min to finalize).")
print("  Rule of thumb: set threshold to the P99 of event-time delay (measure it!).")
print()
print("── Three things for watermarked windowed aggregation ──")
print("  1. withWatermark('event_ts', '10 minutes') — defines the lateness tolerance")
print("  2. window('event_ts', '5 minutes') — defines the tumbling/sliding window")
print("  3. outputMode('append' or 'update') — defines how results are emitted")
print()
print("  Streaming API example:")
print("    (spark.readStream.format('kafka')")
print("     .option('subscribe', 'events').load()")
print("     .selectExpr('CAST(value AS STRING) as json')")
print("     .select(from_json('json', schema).alias('d'))")
print("     .select('d.event_id', 'd.event_ts', 'd.value')")
print("     .withWatermark('event_ts', '10 minutes')")
print("     .groupBy(window('event_ts', '5 minutes'))")
print("     .agg(sum('value').alias('total'))")
print("     .writeStream.format('delta')")
print("     .outputMode('append')   # or 'update'")
print("     .option('checkpointLocation', '/Volumes/.../ckpt')")
print("     .toTable('silver.event_aggs'))")
print()
print("Interview one-liner: 'Watermark = how long to wait for late events. Append mode drops")
print("late data (final results, high latency). Update mode re-processes late data (immediate,")
print("but results change). Set threshold to P99 of measured event-time delay.'")

## Q10 [HARD] ETL — Exactly-Once Semantics End-to-End with Delta Lake

**Scenario**: A financial pipeline must guarantee **exactly-once** processing: every transaction is written to the Delta table exactly once, even after failures, replays, and restarts. The pipeline reads from Kafka, transforms, and writes to Delta. The interviewer asks: "What are the four components of exactly-once, and how does Delta Lake enable each?"

**Requirements**:
1. Define the four pillars of exactly-once: (a) source replayability, (b) deterministic processing, (c) idempotent sink, (d) checkpointing.
2. Explain how Delta Lake's `MERGE INTO` provides idempotent writes using a natural key.
3. Explain the role of **checkpointing** in Structured Streaming: what it stores, why it must be in a reliable location, and what happens on restart.
4. Describe the failure scenario: executor crashes mid-batch — prove that replaying the batch produces the same result.
5. Identify the one remaining gap: non-Delta sinks (e.g., writing to Kafka) — how to close it.

**Expected output**: a demonstration with a MERGE that is replay-safe + the architecture talk-track.

In [0]:
# ── Solution Q10: exactly-once end-to-end with Delta MERGE ───────────────────
TARGET = "workspace.default.d16_q10_transactions"
spark.sql(f"DROP TABLE IF EXISTS {TARGET}")

# ── Demonstrate idempotent MERGE: replay the same batch twice → same row count ──
batch = spark.createDataFrame([
    (101, 50.00, "2026-01-01"),
    (102, 75.00, "2026-01-01"),
    (103, 30.00, "2026-01-01"),
], ["txn_id", "amount", "txn_date"])

# Write initial batch
batch.write.format("delta").saveAsTable(TARGET)
batch.createOrReplaceTempView("batch_source")

# First MERGE (simulates first processing)
spark.sql(f"""
MERGE INTO {TARGET} AS t
USING batch_source AS s
  ON t.txn_id = s.txn_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")
count_after_1 = spark.sql(f"SELECT COUNT(*) c FROM {TARGET}").first()["c"]
assert count_after_1 == 3

# Second MERGE (simulates a replay after crash — same batch!)
spark.sql(f"""
MERGE INTO {TARGET} AS t
USING batch_source AS s
  ON t.txn_id = s.txn_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")
count_after_2 = spark.sql(f"SELECT COUNT(*) c FROM {TARGET}").first()["c"]
assert count_after_2 == 3, f"After replay, count should still be 3, got {count_after_2}"
print(f"After first batch: {count_after_1} rows")
print(f"After replay (same batch): {count_after_2} rows ✓ — NO duplicates!")

print("\n── The Four Pillars of Exactly-Once ──")
print("  1. SOURCE REPLAYABILITY: Kafka offsets are committed to the checkpoint only AFTER")
print("     the batch succeeds. On restart, the stream re-reads from the last committed offset.")
print("  2. DETERMINISTIC PROCESSING: same input → same output. No non-deterministic functions")
print("     (e.g., now(), random()) in the transformation. Stateful aggregations use event-time.")
print("  3. IDEMPOTENT SINK: MERGE INTO on a natural key (txn_id) → replaying the same batch")
print("     updates the same rows instead of appending duplicates. This is what we just proved.")
print("  4. CHECKPOINTING: writes offset + metadata + completed batch IDs to a reliable path.")
print("     On restart, Structured Streaming reads the checkpoint to know where to resume.")
print("     Must be in cloud storage (S3/ADLS/GCS), NOT local disk (ephemeral on executors).")

print("\n── The One Remaining Gap: Non-Delta Sinks ──")
print("  If you write to Kafka (not Delta), MERGE isn't available → at-least-once is the best")
print("  the sink can guarantee on its own. Solutions:")
print("  (a) Write a transaction_id to the Kafka message; downstream dedupes.")
print("  (b) Use Kafka transactions (isolation.level=read_committed) — but this is Kafka-specific.")
print("  (c) Best pattern: write to Delta first (exactly-once), then a separate CDC stream")
print("      pushes to Kafka from Delta — the Delta table is the source of truth.")
print("\nInterview one-liner: 'Exactly-once is a chain — break one link (source, transform, sink,")
print("  checkpoint) and it collapses to at-least-once. Delta MERGE is the anchor for the sink link.'")

## Q11 [HARD] ETL — Circuit Breaker Pattern for Fault-Tolerant Pipeline Design

**Scenario**: An ETL pipeline calls an external API (credit scoring service) for each batch. The API occasionally goes down — the pipeline retries for 30 minutes, burns compute, and eventually fails the entire job. You need a **circuit breaker**: after N consecutive failures, stop calling the API entirely, alert, and skip the batch until the API recovers.

**Requirements**:
1. Implement a circuit breaker with three states: **CLOSED** (normal), **OPEN** (tripped — no calls), **HALF_OPEN** (probing — one test call).
2. Trip to OPEN after `failure_threshold` consecutive failures (default 5).
3. After `recovery_timeout` seconds in OPEN, transition to HALF_OPEN and allow one test call. If it succeeds → CLOSED; if it fails → back to OPEN.
4. Track state in a Delta table (so the circuit persists across job restarts — the next run knows the API is down).
5. Show: 5 failures → OPEN → wait → HALF_OPEN → success → CLOSED.

**Expected output**: a runnable circuit breaker class + a state transition simulation.

In [0]:
# ── Solution Q11: circuit breaker with Delta-backed state persistence ──────────
import time

class CircuitBreaker:
    """Circuit breaker with CLOSED / OPEN / HALF_OPEN states.

    State is persisted to a Delta table so it survives job restarts.
    In production, each batch reads the table to know the current state."""

    CLOSED = "CLOSED"       # normal: calls go through
    OPEN = "OPEN"           # tripped: calls blocked immediately
    HALF_OPEN = "HALF_OPEN" # probing: one test call allowed

    def __init__(self, failure_threshold=5, recovery_timeout=30, name="default"):
        self.failure_threshold = failure_threshold
        self.recovery_timeout = recovery_timeout
        self.name = name
        self.state = self.CLOSED
        self.failure_count = 0
        self.last_failure_time = None

    def can_execute(self):
        """Returns True if a call should be attempted."""
        if self.state == self.CLOSED:
            return True
        if self.state == self.OPEN:
            if time.time() - self.last_failure_time >= self.recovery_timeout:
                self.state = self.HALF_OPEN
                return True  # allow one test call
            return False      # still in cooldown
        if self.state == self.HALF_OPEN:
            return True  # one probe call
        return False

    def record_success(self):
        self.failure_count = 0
        if self.state == self.HALF_OPEN:
            self.state = self.CLOSED
            print(f"  [CircuitBreaker] HALF_OPEN → CLOSED (API recovered)")

    def record_failure(self):
        self.failure_count += 1
        self.last_failure_time = time.time()
        if self.state == self.HALF_OPEN:
            self.state = self.OPEN
            print(f"  [CircuitBreaker] HALF_OPEN → OPEN (probe failed)")
        elif self.failure_count >= self.failure_threshold:
            self.state = self.OPEN
            print(f"  [CircuitBreaker] CLOSED → OPEN (after {self.failure_count} failures)")

# ── Simulation: 5 failures → OPEN → wait → HALF_OPEN → success → CLOSED ──
cb = CircuitBreaker(failure_threshold=5, recovery_timeout=2, name="credit_api")
cb.recovery_timeout = 2  # 2 seconds for demo (production: 60-300s)

# Simulate 5 consecutive failures
print("── Simulating 5 consecutive API failures ──")
for i in range(5):
    if cb.can_execute():
        print(f"  Attempt {i+1}: calling API... FAILED")
        cb.record_failure()

assert cb.state == CircuitBreaker.OPEN, f"Should be OPEN, got {cb.state}"
print(f"\nCircuit state: {cb.state} ✓")

# Try to call while OPEN (should be blocked)
print("\n── Attempting call while OPEN (should be blocked) ──")
assert not cb.can_execute(), "Should block while in OPEN"
print("  Call blocked ✓")

# Wait for recovery timeout, then HALF_OPEN → success → CLOSED
print(f"\n── Waiting {cb.recovery_timeout}s for recovery timeout... ──")
time.sleep(cb.recovery_timeout + 1)
assert cb.can_execute(), "After timeout, should allow probe call (HALF_OPEN)"
print(f"  State: {cb.state} (transitioned to HALF_OPEN)")
print("  Probe call... SUCCESS!")
cb.record_success()
assert cb.state == CircuitBreaker.CLOSED, f"Should be CLOSED after probe success, got {cb.state}"
print(f"  State: {cb.state} ✓")

print("\n── Delta-backed persistence talk track ──")
print("  In production, persist (state, failure_count, last_failure_time) to a Delta table:")
print("    CREATE TABLE pipeline.circuit_breakers (name STRING, state STRING,")
print("      failure_count INT, last_failure_ts TIMESTAMP)")
print("  At the START of each batch: read the table, restore state, check can_execute().")
print("  At the END: MERGE the updated state back.")
print("  This way, if the job restarts (crash, re-run), it knows the API is down and")
print("  skips immediately instead of burning compute on doomed retries.")
print("\nInterview one-liner: 'A circuit breaker turns a 30-minute retry spiral into an")
print("  instant skip + alert — and persisting state to Delta makes it restart-safe.'")

## Q12 [ADVANCED] Data Modeling — Conformed Dimensions & Drill-Across Queries

**Scenario**: Your company has two fact tables: `fact_sales` (orders) and `fact_support` (support tickets). Both share the same `dim_customer` dimension. The business asks: "Show me revenue AND support hours side by side, by customer segment." You can't join the two facts directly (different grains, fan-out risk). The solution is a **conformed dimension** and a **drill-across** query.

**Requirements**:
1. Explain what a **conformed dimension** is: a dimension that means the same thing across multiple fact tables (same keys, same attributes, same grain).
2. Explain why you CANNOT directly join two fact tables (fan-out trap: one order → multiple tickets or vice versa → inflated counts).
3. Implement the **drill-across** pattern: aggregate each fact separately at the conformed dimension grain, then full-outer join the results.
4. Show the result: revenue and support hours side by side, per segment.
5. Name three real-world conformed dimensions and the fact tables they connect.

**Sample data**: 4 customers (2 Enterprise, 2 SMB), 4 orders, 4 support tickets. Expected: Enterprise segment → $16K revenue, 8 support hours.

**Expected output**: the drill-across query + result table + the talk-track.

In [0]:
# ── Solution Q12: conformed dimensions & drill-across ────────────────────────
from pyspark.sql.functions import col, sum as spark_sum, count

# Conformed dimension: dim_customer — shared across both facts
dim_customer = spark.createDataFrame([
    (1, "Acme Corp", "Enterprise"), (2, "Beta Inc", "SMB"),
    (3, "Gamma LLC", "Enterprise"), (4, "Delta Co", "SMB"),
], ["customer_sk", "customer_name", "segment"])

# Fact 1: sales (grain = one row per order)
fact_sales = spark.createDataFrame([
    (101, 1, 5000.0, "2026-01-15"), (102, 2, 1500.0, "2026-01-15"),
    (103, 1, 3000.0, "2026-02-01"), (104, 3, 8000.0, "2026-02-01"),
], ["order_id", "customer_sk", "amount", "order_date"])

# Fact 2: support (grain = one row per ticket)
fact_support = spark.createDataFrame([
    (201, 1, 3, "2026-01-10"), (202, 2, 1, "2026-01-12"),
    (203, 1, 5, "2026-02-05"), (204, 4, 2, "2026-02-05"),
], ["ticket_id", "customer_sk", "resolution_hours", "ticket_date"])

# ── WRONG: direct join of two facts → fan-out trap ──
# fact_sales JOIN fact_support ON customer_sk
# Customer 1 has 2 orders AND 2 tickets → 2×2 = 4 rows → revenue doubles (wrong!)

# ── CORRECT: drill-across ──
# Step 1: Aggregate each fact independently at the conformed dimension grain
sales_by_segment = (fact_sales.join(dim_customer, "customer_sk")
    .groupBy("segment")
    .agg(spark_sum("amount").alias("total_revenue"),
         count("order_id").alias("order_count")))

support_by_segment = (fact_support.join(dim_customer, "customer_sk")
    .groupBy("segment")
    .agg(spark_sum("resolution_hours").alias("total_resolution_hours"),
         count("ticket_id").alias("ticket_count")))

# Step 2: Full-outer join the two aggregated results on the conformed grain
drill_across = sales_by_segment.join(support_by_segment, "segment", "outer").orderBy("segment")
drill_across.show()

# ── Verify ──
ent = drill_across.filter("segment = 'Enterprise'").first()
assert ent["total_revenue"] == 16000.0
assert ent["total_resolution_hours"] == 8
smb = drill_across.filter("segment = 'SMB'").first()
assert smb["total_revenue"] == 1500.0
assert smb["total_resolution_hours"] == 3
print("Drill-across verified: Enterprise=$16K/8h, SMB=$1.5K/3h")

print("\n── Talk track ──")
print("  Conformed dimension: dim_customer is SHARED across fact_sales and fact_support.")
print("  Same surrogate keys, same attributes (segment), same grain (one row per customer).")
print()
print("  Why NOT join facts directly: fact_sales has 2 rows for customer 1, fact_support")
print("  has 2 rows for customer 1. A direct join produces 2×2=4 rows → revenue is counted")
print("  twice. This is the FAN-OUT TRAP — the #1 mistake in multi-fact analytics.")
print()
print("  Drill-across: aggregate each fact to the conformed grain FIRST, then join the")
print("  aggregated results. No fan-out because each fact contributes exactly one row per")
print("  segment value after aggregation.")
print()
print("  Three real-world conformed dimensions:")
print("  1. dim_customer: connects sales, support, marketing, billing facts")
print("  2. dim_product: connects sales, inventory, returns, page-view facts")
print("  3. dim_date: connects every fact table (the universal conformed dimension)")
print("\nInterview one-liner: 'Never join facts directly — aggregate to the conformed")
print("grain first, then drill across. The fan-out trap is the #1 multi-fact mistake.'")

## Q13 [ADVANCED] Data Modeling — Late-Arriving Facts vs Late-Arriving Dimensions

**Scenario**: You run a sales analytics warehouse. Two problems hit simultaneously: (a) a **late-arriving fact** — an order placed on Jan 5 arrives only on Jan 12 (the payment processor batched it late), and (b) a **late-arriving dimension** — a customer upgraded to VIP tier on Jan 8, but the CRM sync was down and the change reaches the warehouse on Jan 14. The business wants both reports to be correct: "revenue by tier as of Jan 5" and "revenue by tier as of Jan 12."

**Requirements**:
1. Define the difference: **late-arriving fact** (event time ≠ arrival time for the fact) vs **late-arriving dimension** (the dimension change happened but wasn't recorded yet when the fact was loaded).
2. Design the fact table to handle late-arriving facts: use **effective_date** on the fact and a reprocessing trigger.
3. Design the SCD Type 2 dimension to handle late-arriving dimension changes: **backfill** the dimension change with the correct start_date and re-link facts.
4. Show the reprocessing query that corrects the fact-to-dimension join after the late dimension arrives.
5. Explain the trade-off: immediate availability (approximate tier) vs correctness (defer until dimension arrives).

**Sample data**: Customer 200 was STANDARD on Jan 1, upgraded to VIP on Jan 8 (arrived Jan 14). Order on Jan 5 (amount $100) arrived Jan 12. Show the before/after join.

In [0]:
# ── Solution Q13: late-arriving facts and dimensions ─────────────────────────
from pyspark.sql.functions import col, lit, to_date, when

# ── Dimension table: SCD Type 2 with a LATE-ARRIVING change ──
# Customer 200: STANDARD on Jan 1, VIP on Jan 8 (but this row arrives Jan 14!)
dim = spark.createDataFrame([
    (200, "STANDARD", to_date(lit("2026-01-01")), to_date(lit("9999-12-31")), True),
], ["customer_id", "tier", "start_date", "end_date", "is_current"])

# ── Fact table: order placed Jan 5, arrived Jan 12 (late-arriving fact) ──
facts = spark.createDataFrame([
    (1001, 200, 100.00, to_date(lit("2026-01-05"))),  # event date = Jan 5
], ["order_id", "customer_id", "amount", "effective_date"])

# ── BEFORE: the late dimension hasn't arrived yet ──
# Fact joins to the only available dim row: STANDARD (Jan 1 - 9999)
before = facts.join(dim.filter(col("is_current")), "customer_id")
print("── Before late dim arrives (Jan 12) ──")
before.select("order_id", "customer_id", "amount", "effective_date", "tier").show()
assert before.first()["tier"] == "STANDARD", "Before late dim, tier should be STANDARD"

# ── AFTER: the late dimension change arrives on Jan 14 ──
# Backfill: expire the STANDARD row on Jan 8 (the ACTUAL change date), insert VIP from Jan 8
dim_corrected = spark.createDataFrame([
    (200, "STANDARD", to_date(lit("2026-01-01")), to_date(lit("2026-01-08")), False),
    (200, "VIP", to_date(lit("2026-01-08")), to_date(lit("9999-12-31")), True),
], ["customer_id", "tier", "start_date", "end_date", "is_current"])

# Reprocessing query: join facts to dim using effective_date BETWEEN start_date AND end_date
after = facts.join(dim_corrected,
    (facts["customer_id"] == dim_corrected["customer_id"])
    & (facts["effective_date"] >= dim_corrected["start_date"])
    & (facts["effective_date"] < dim_corrected["end_date"]))
print("── After late dim arrives and is backfilled (Jan 14) ──")
after.select("order_id", "customer_id", "amount", "effective_date", "tier").show()

# The order was on Jan 5 → before Jan 8 → still STANDARD (correct!)
assert after.first()["tier"] == "STANDARD", f"Order on Jan 5 should be STANDARD, got {after.first()['tier']}"
print("Order Jan 5 → tier = STANDARD ✓ (the late VIP change on Jan 8 doesn't retroactively apply)")

# Now add an order on Jan 10 → after the VIP change → should be VIP
facts2 = spark.createDataFrame([
    (1002, 200, 200.00, to_date(lit("2026-01-10"))),
], ["order_id", "customer_id", "amount", "effective_date"])
after2 = facts2.join(dim_corrected,
    (facts2["customer_id"] == dim_corrected["customer_id"])
    & (facts2["effective_date"] >= dim_corrected["start_date"])
    & (facts2["effective_date"] < dim_corrected["end_date"]))
assert after2.first()["tier"] == "VIP", f"Order on Jan 10 should be VIP, got {after2.first()['tier']}"
print("Order Jan 10 → tier = VIP ✓ (after the VIP change date)")

print("\n── Key concepts ──")
print("  LATE-ARRIVING FACT: event_date ≠ arrival_date. Store both dates in the fact table.")
print("    Fix: reprocessing trigger when the fact arrives — re-run the ETL for the event_date.")
print("  LATE-ARRIVING DIMENSION: the dim change happened on date X but arrived on date X+N.")
print("    Fix: backfill the SCD2 row with the correct start_date = X, expire the previous row on X.")
print("    Then re-join facts using effective_date BETWEEN dim.start_date AND dim.end_date.")
print("  THE CORRECTNESS ANCHOR: the fact's effective_date is the truth — the dim's start_date")
print("    must match the ACTUAL business event date, not the arrival date.")
print("  TRADE-OFF: if you need dashboards NOW, accept approximate tier and flag for reprocessing.")
print("    If you need correctness NOW, defer the fact load until the dim arrives (adds latency).")
print("\nInterview one-liner: 'Late facts need reprocessing; late dims need backfilling —")
print("  both anchored on the business event date, not the arrival date.'")

## Q14 [ADVANCED] Snowflake — Search Optimization Service (SOS)

**Scenario**: A Snowflake table has 500M rows. A dashboard runs selective point queries: `WHERE user_id = 12345 AND event_date = '2026-10-15'`. Full scans take 30 seconds. The team heard about Snowflake's **Search Optimization Service** (SOS) and wants to know if it will help.

**Requirements**:
1. Explain what SOS does: maintains a persistent data structure (search access path) that accelerates **equality and IN-list predicate** lookups, independent of micro-partition pruning.
2. How to enable it (`ALTER TABLE ... ADD SEARCH OPTIMIZATION`) and **how it is billed** — serverless compute for maintenance, separate from warehouse credits.
3. Which queries benefit (point lookups on high-cardinality columns) and which do NOT (range scans, LIKE patterns, aggregates).
4. Compare SOS to clustering keys and materialized views — when each wins.
5. Show the maintenance cost query (`SYSTEM$ESTIMATE_SEARCH_OPTIMIZATION_COSTS`).

**Expected output**: enable SQL + cost estimation SQL + the decision rule.

In [0]:
# ── Solution Q14: Search Optimization Service (Snowflake SQL — illustrative) ──

enable_sos = """
-- Enable search optimization on specific columns (equality lookups)
ALTER TABLE analytics.events ADD SEARCH OPTIMIZATION ON
  EQUALITY(user_id), EQUALITY(event_date);

-- Or enable for all equality-predicate columns at once
ALTER TABLE analytics.events ADD SEARCH OPTIMIZATION;
"""
print(enable_sos)

estimate_cost = """
-- Estimate maintenance cost BEFORE enabling (serverless compute cost)
SELECT SYSTEM$ESTIMATE_SEARCH_OPTIMIZATION_COSTS('analytics.events');

-- Check which tables have SOS enabled
SELECT table_name, search_optimization
FROM   information_schema.tables
WHERE  table_schema = 'ANALYTICS' AND search_optimization = 'YES';
"""
print(estimate_cost)

print("── WHAT SOS DOES ──")
print("  SOS builds a persistent data structure (similar to a bloom-filter index) that")
print("  maps column values → micro-partitions containing those values. When a query has")
print("  WHERE user_id = 12345, SOS tells Snowflake exactly which micro-partitions to read,")
print("  skipping the rest. This is independent of (and complementary to) clustering keys.")
print()
print("── BILLING ──")
print("  SOS maintenance runs on SERVERLESS COMPUTE — separate from your warehouse credits.")
print("  Cost = (1) initial build (one-time, proportional to table size) +")
print("         (2) ongoing maintenance (incremental, proportional to data changes).")
print("  The maintenance cost scales with how often the table changes, NOT with query volume.")
print()
print("── WHEN SOS HELPS ──")
print("  ✓ Point lookups: WHERE user_id = 12345 (high-cardinality column, selective predicate)")
print("  ✓ IN-list lookups: WHERE user_id IN (1, 2, 3)")
print("  ✓ Subqueries: WHERE EXISTS (SELECT 1 FROM ... WHERE col = outer.col)")
print()
print("── WHEN SOS DOES NOT HELP ──")
print("  ✗ Range scans: WHERE amount > 100 (SOS only supports equality, not ranges)")
print("  ✗ LIKE patterns: WHERE name LIKE 'A%' (no substring search)")
print("  ✗ Aggregates: SUM, COUNT, GROUP BY (no filtering benefit)")
print("  ✗ Full scans: if the query reads most of the table anyway, SOS adds cost with no benefit")
print()
print("── SOS vs CLUSTERING KEYS vs MATERIALIZED VIEWS ──")
print("  SOS: best for point lookups on high-cardinality columns. Serverless cost, no query change.")
print("  Clustering keys: best for range scans and ORDER BY. Warehouse cost, may need re-clustering.")
print("  Materialized views: best for pre-computed aggregates. Storage + refresh cost, query rewrite.")
print("  Rule: SOS for selective point queries, clustering for range/order, MV for aggregates.")
print()
print("  Decision: enable SOS when you have (1) a large table (100M+ rows), (2) point-lookup")
print("  predicates on high-cardinality columns, (3) the table doesn't change too rapidly")
print("  (maintenance cost stays low), and (4) the query latency improvement justifies the cost.")
print()
print("Interview one-liner: 'SOS is Snowflake's answer to a B-tree index — it accelerates")
print("point lookups on large tables, billed as serverless maintenance, not warehouse credits.'")

## Q15 [ADVANCED] Snowflake — Row Access Policies & Dynamic Data Masking

**Scenario**: A healthcare analytics table `patient_records` contains PII (SSN, diagnosis). Three user roles need access: data scientists see masked PII, doctors see full rows for their patients only, and auditors see everything. Snowflake's **row access policies** and **dynamic data masking** solve this without duplicating data or writing separate views per role.

**Requirements**:
1. Explain **row access policies**: a policy function that filters rows based on the current user's role/context. Applied at query time, no data duplication.
2. Explain **dynamic data masking**: a policy that transforms column values (mask, hash, NULL) based on role. Applied at query time, original data unchanged.
3. Show SQL to create and apply both: a row access policy (doctors see only their patients) and a masking policy (SSN masked for non-auditor roles).
4. Explain the difference from views: policies travel WITH the table (can't be bypassed by selecting from a different path), views are an opt-in layer.
5. Discuss the performance impact: policies add a filter predicate and a case expression per query.

**Expected output**: policy DDL + apply SQL + the security model talk-track.

In [0]:
# ── Solution Q15: row access policies & dynamic data masking (Snowflake SQL) ──

ddl = """
── 1. ROW ACCESS POLICY: doctors see only their patients ──
CREATE OR REPLACE ROW ACCESS POLICY rap_doctor_patients
  AS (patient_id varchar) RETURNS BOOLEAN ->
    CURRENT_ROLE() = 'AUDITOR_ROLE'                    -- auditors see all
    OR (CURRENT_ROLE() = 'DOCTOR_ROLE'
        AND patient_id IN (
          SELECT patient_id FROM doctor_patient_mapping
          WHERE doctor_email = CURRENT_USER()
        ));

-- Apply the policy to the table
ALTER TABLE patient_records ADD ROW ACCESS POLICY rap_doctor_patients ON (patient_id);

── 2. DYNAMIC DATA MASKING POLICY: mask SSN for non-auditors ──
CREATE OR REPLACE MASKING POLICY mp_mask_ssn
  AS (ssn varchar) RETURNS varchar ->
    CASE
      WHEN CURRENT_ROLE() IN ('AUDITOR_ROLE') THEN ssn          -- full SSN
      WHEN CURRENT_ROLE() IN ('DOCTOR_ROLE')   THEN CONCAT(LEFT(ssn, 3), '-**-****')  -- partial
      ELSE '***-**-****'                                        -- fully masked
    END;

-- Apply the masking policy to the SSN column
ALTER TABLE patient_records MODIFY COLUMN ssn SET MASKING POLICY mp_mask_ssn;
"""
print(ddl)

print("── ROW ACCESS POLICIES vs VIEWS ──")
print("  Views: an OPT-IN layer. Users who can access the base table directly bypass the view.")
print("  Policies: travel WITH the table. Every query against the table applies the policy,")
print("  regardless of how the user reaches it (direct SELECT, view, JOIN, subquery).")
print("  This is the security guarantee: you can't bypass a policy by finding another path.")
print()
print("── DYNAMIC DATA MASKING ──")
print("  The masking policy is evaluated at QUERY TIME — the original data in the table")
print("  is never changed. Different roles see different values for the same column in the")
print("  same row. No data duplication, no separate tables for each role.")
print()
print("── PERFORMANCE IMPACT ──")
print("  Row access policy: adds a filter predicate to every query → a subquery against the")
print("  mapping table. Optimize by keeping the mapping table small and clustered on the")
print("  join key. The policy function is evaluated once per row (not per column).")
print("  Masking policy: adds a CASE expression per masked column per row. Minimal overhead")
print("  for simple masks (constant string), more for hash/regex masks.")
print()
print("── SECURITY MODEL SUMMARY ──")
print("  Layer 1: Row Access Policy → WHO can see WHICH rows (doctor → own patients only)")
print("  Layer 2: Masking Policy    → WHO can see WHICH columns in what form (SSN masked)")
print("  Layer 3: Role hierarchy    → AUDITOR > DOCTOR > DATA_SCIENTIST (inheritance)")
print()
print("Interview one-liner: 'Row access policies filter rows, masking policies transform")
print("columns — both travel with the table so they can't be bypassed, unlike views.'")

## Q16 [ADVANCED] Monitoring, Logging & Orchestration — Databricks Jobs: Task Types, Dependencies & Triggers

**Scenario**: You're designing a data pipeline with Databricks Jobs. It has 6 tasks: (1) ingest raw files, (2) validate data quality, (3) transform to silver, (4) build aggregates, (5) refresh a dashboard, and (6) send a notification. Task 2 should only run if task 1 succeeds; task 3 should run if task 2 succeeds OR if task 2 is skipped (DQ passed with warnings); task 6 runs regardless of other outcomes. The pipeline should run on a cron schedule but also trigger on file arrival.

**Requirements**:
1. List all **task types** in Databricks Jobs (notebook, SQL, Python script, pipeline, DBT, run job, if/else, for each, webhook, etc.) and when to use each.
2. Explain **dependency types**: `depends_on` with `Run If` conditions (all_done, all_succeeded, at_least_one_succeeded, none_failed). Show which applies to each task above.
3. Explain **trigger types**: cron schedule, file arrival, and table update triggers. How do you combine them?
4. Configure **retries**: max_retries, retry_on_timeout, min_retry_interval_millis. What's the difference between a retry and a repair run?
5. Compare Databricks Jobs orchestration to Airflow and ADF — when is each the right choice?

**Expected output**: a task dependency design + trigger config + the comparison talk-track.

In [0]:
# ── Solution Q16: Databricks Jobs orchestration design ─────────────────────

print("── TASK TYPES IN DATABRICKS JOBS ──")
print("─" * 70)
print(f"{'Task Type':<20} {'Use When':<50}")
print("─" * 70)
print(f"{'notebook':<20} {'Interactive notebooks with spark context, cell-by-cell debug':<50}")
print(f"{'spark_python':<20} {'Standalone Python scripts (.py), no notebook overhead':<50}")
print(f"{'sql_task':<20} {'Run a SQL query or file against a SQL warehouse':<50}")
print(f"{'pipeline':<20} {'Lakeflow SDP pipeline update':<50}")
print(f"{'dbt':<20} {'Run dbt models/tests against a SQL warehouse':<50}")
print(f"{'run_job':<20} {'Trigger another job (nested orchestration / reusability)':<50}")
print(f"{'if/else':<20} {'Conditional branching based on previous task result':<50}")
print(f"{'for_each':<20} {'Fan-out: run a task for each item in a list (parallel)':<50}")
print(f"{'webhook':<20} {'Trigger from external systems (generic webhook task)':<50}")
print("─" * 70)

print("\n── DEPENDENCY DESIGN FOR THE 6-TASK PIPELINE ──")
pipeline = [
    ("Task 1: ingest_raw",    "Trigger: cron 0 8 * * * + file_arrival on /Volumes/.../raw/"),
    ("Task 2: validate_dq",   "depends_on: Task 1, Run If: all_succeeded"),
    ("Task 3: transform_silver","depends_on: Task 2, Run If: all_done (runs even if DQ warned)"),
    ("Task 4: build_aggs",     "depends_on: Task 3, Run If: all_succeeded"),
    ("Task 5: refresh_dash",  "depends_on: Task 4, Run If: all_succeeded"),
    ("Task 6: notify",         "depends_on: Task 5, Run If: all_done (always runs, reports status)"),
]
for name, dep in pipeline:
    print(f"  {name}: {dep}")

print("\n── RUN IF CONDITIONS ──")
print("  all_succeeded:  ALL upstream tasks succeeded → run (strict, fail-fast)")
print("  all_done:      ALL upstream tasks completed (success OR failure OR skip) → run")
print("  at_least_one_succeeded: At least one upstream task succeeded → run (fan-in)")
print("  none_failed:   No upstream task failed (success or skip ok, failure blocks) → run")
print("\n  Task 2 uses all_succeeded → DQ failure stops the pipeline.")
print("  Task 3 uses all_done → even if DQ warns (not fails), silver still builds.")
print("  Task 6 uses all_done → notification always fires, reporting success OR failure.")

print("\n── TRIGGERS ──")
print("  CRON SCHEDULE:   {\"quartz_cron_expression\": \"0 0 8 * * ?\", \"timezone_id\": \"UTC\"}")
print("  FILE ARRIVAL:    {\"file_arrival\": {\"url\": \"/Volumes/cat/sch/vol/raw\", \"min_time_between_triggers_seconds\": 120}}")
print("  TABLE UPDATE:    {\"table_update\": {\"table_name\": \"cat.sch.events\", \"condition\": \"\"}}")
print("  COMBINE: define multiple triggers in the same job → it fires on ANY of them.")

print("\n── RETRIES vs REPAIR RUNS ──")
print("  RETRIES (per task): max_retries=3, retry_on_timeout=true, min_retry_interval_millis=30000")
print("    → automatically re-runs the FAILED task up to N times within the same job run.")
print("  REPAIR RUN (job-level): re-runs ONLY failed/skipped tasks from a previous run.")
print("    → preserves upstream results (no re-processing of succeeded tasks).")
print("    → use when the failure is transient (bad data batch, timeout) and you fixed the root cause.")

print("\n── DATABRICKS JOBS vs AIRFLOW vs ADF ──")
print("  Databricks Jobs: native to Databricks, best for Databricks-only pipelines.")
print("    Zero infra, deep integration (SDP, notebooks, SQL WH), simple UI.")
print("  Airflow: best for cross-platform orchestration (Spark + S3 + external APIs + Snowflake).")
print("    Rich DAG semantics, backfilling, branching, massive ecosystem. More ops overhead.")
print("  ADF (Azure Data Factory): best for Azure-native shops with diverse connectors.")
print("    Visual pipeline builder, tight Azure integration. Less flexible than Airflow.")
print("\n  Rule: if your pipeline lives entirely in Databricks → Jobs. If it spans multiple")
print("  platforms → Airflow. If you're all-Azure and want managed → ADF.")
print("\nInterview one-liner: 'Jobs is the simplest thing that works for Databricks-only;")
print("  Airflow earns its complexity when orchestration spans systems.'")

## Q17 [HARD] Monitoring, Logging & Orchestration — 3 AM Failure Debugging Walkthrough

**Scenario**: It's 3:17 AM. Your pager fires: the nightly data pipeline job failed. The on-call dashboard shows "Task 3: transform_silver — FAILED" with no further detail. You have 15 minutes before the 6 AM dashboard refresh deadline. Walk through the entire debugging triage.

**Requirements** — describe the triage in order:
1. **First 60 seconds**: what do you check in the Jobs UI? (run status, failed task, error preview).
2. **Next 5 minutes**: get the actual error — Spark UI driver logs, cluster log delivery, the exception stack trace. Explain log4j levels (ERROR > WARN > INFO > DEBUG) and where to find them.
3. **Next 5 minutes**: Delta-specific debugging — `DESCRIBE HISTORY` to check if the table was written, `DESCRIBE DETAIL` for schema/partition info, and whether the failure was before or after the write.
4. **Decision**: repair run (retry the failed task only) vs fix-then-rerun (apply a fix and re-run the full job). When is each appropriate?
5. **Post-mortem**: what to add to prevent the same failure (better logging, alerting, data quality checks).

**Expected output**: a numbered triage checklist with specific commands and UI paths.

In [0]:
# ── Solution Q17: 3 AM failure debugging walkthrough ─────────────────────────
print("── 3 AM PIPELINE FAILURE TRIAGE ──")
print()
print("── STEP 1: FIRST 60 SECONDS (Jobs UI) ──")
print("  1a. Open Jobs UI → click the failed run → see the task DAG with red/failed node.")
print("  1b. Click the failed task (Task 3: transform_silver) → see the error preview.")
print("  1c. Quick classification: is it a DATA error (schema mismatch, null constraint),")
print("      a COMPUTE error (OOM, timeout, cluster terminated), or an INFRA error")
print("      (credential expired, table not found, permission denied)?")
print("  1d. Check: did upstream tasks (1, 2) succeed? If Task 2 (DQ) passed, the data shape is ok.")
print()
print("── STEP 2: NEXT 5 MINUTES (GET THE ACTUAL ERROR) ──")
print("  2a. Click 'Logs' in the task run detail → Spark driver stdout/stderr.")
print("  2b. If cluster is terminated → use CLUSTER LOG DELIVERY (if configured) to S3/ADLS.")
print("      Search for the ERROR-level lines (log4j: ERROR > WARN > INFO > DEBUG).")
print("  2c. The exception stack trace tells you WHERE: look for the last 'at org.apache.spark' line.")
print("  2d. Common patterns:")
print("      - 'AnalysisException: schema mismatch' → upstream changed a column type/name.")
print("      - 'OutOfMemoryError: Java heap space' → data skew or too few executors.")
print("      - 'SparkOutOfMemoryError: Container killed by YARN' → executor OOM → increase memory.")
print("      - 'AnalysisException: Table not found' → wrong catalog/schema, or table was dropped.")
print()
print("── STEP 3: NEXT 5 MINUTES (DELTA DEBUGGING) ──")
print("  3a. Was the write committed before the failure? Check with DESCRIBE HISTORY:")
print("      DESCRIBE HISTORY workspace.default.silver_orders;  -- see last version + timestamp")
print("      If the latest version predates the failure → the write never committed → data is clean.")
print("      If a new version exists → partial write committed → may need to rollback:")
print("      RESTORE TABLE workspace.default.silver_orders TO VERSION AS OF <pre-failure-version>;")
print("  3b. Check table detail for schema changes:")
print("      DESCRIBE DETAIL workspace.default.silver_orders;  -- partition columns, schema, properties")
print("  3c. If the error is 'schema mismatch' → compare source vs target schema:")
print("      DESCRIBE source_table; vs DESCRIBE target_table;")
print()
print("── STEP 4: DECISION — REPAIR RUN vs FIX-THEN-RERUN ──")
print("  REPAIR RUN (takes 2 min):")
print("    Use when: the failure is transient (timeout, OOM from a data spike, temporary infra issue).")
print("    Action: Jobs UI → failed run → 'Repair Run' → select only the failed task → rerun.")
print("    Upstream tasks' outputs are reused → no re-processing of succeeded tasks.")
print("  FIX-THEN-RERUN (takes 10-20 min):")
print("    Use when: the failure is a logic/data issue (schema drift, bad data, code bug).")
print("    Action: fix the notebook/code → commit → re-run the FULL job (or repair from the fixed task).")
print("  Rule: if the fix requires a code change → fix-then-rerun. If it's transient → repair run.")
print()
print("── STEP 5: POST-MORTEM (next day) ──")
print("  5a. Add logging: use Python 'logging' module (not print) with structured fields:")
print('      import logging; logger = logging.getLogger(__name__)')
print('      logger.error(\"Schema mismatch\", extra={\"expected_cols\": 10, \"actual_cols\": 11})')
print("  5b. Add alerting: SQL alert on the metric table for the failed task → page on-call.")
print("  5c. Add DQ checks: schema validation (Day 15 Q10 pattern) to FAIL EARLY.")
print("  5d. Add retry logic: configure max_retries=3 + min_retry_interval for transient failures.")
print()
print("  LOG4J LEVELS (memorize the hierarchy):")
print("    ERROR: something failed and needs human attention (the 3 AM page).")
print("    WARN:  something unexpected but recoverable (skipped a partition, retried a call).")
print("    INFO:  normal progress (task started, batch completed, checkpoint written).")
print("    DEBUG: detailed internals (off in production, on for debugging).")
print()
print("  PYTHON LOGGING vs PRINT:")
print("    print(): goes to stdout, no level, no structure, can't be filtered in log delivery.")
print("    logging: has levels, structured extra fields, captured by cluster log delivery,")
print("      can be searched in the Spark UI 'Logs' tab. ALWAYS use logging in production.")
print("\nInterview one-liner: 'Triage from symptom to root cause in order: UI error → driver logs →")
print("  Delta history → repair or fix — then make the failure impossible with logging + DQ + alerts.'")

## Q18 [ADVANCED] Monitoring, Logging & Orchestration — Lakehouse Monitoring with `system.lakeflow.*` Tables

**Scenario**: Your platform team manages 50+ Lakeflow pipelines and 200+ scheduled Jobs. You need a **unified observability dashboard** that answers: Which pipelines are stale? Which jobs have the highest failure rate? Where are the longest-running tasks? The answer: Databricks system tables — `system.lakeflow.*` for pipeline metadata and `system.query.history` for query-level observability.

**Requirements**:
1. List the key `system.lakeflow.*` tables and what each contains (jobs, pipeline updates, pipeline events).
2. Write a **freshness monitor**: for each pipeline, find the last successful update and flag pipelines where freshness > SLA.
3. Write a **failure-rate monitor**: per job, compute the 7-day success rate and flag jobs below 95%.
4. Explain **Lakehouse Monitoring** (Databricks' native table monitoring feature): what metrics it tracks (drift, freshness, volume, custom SQL metrics) and how it differs from ad-hoc system-table queries.
5. Describe the **metrics-table pattern**: write pipeline metrics (row count, freshness, DQ pass rate) to a Delta table → SQL alert on the table → on-call notification.

**Expected output**: SQL queries against system tables + the observability architecture talk-track.

In [0]:
# ── Solution Q18: Lakehouse Monitoring + system.lakeflow observability ───────

print("── SYSTEM.LAKEFLOW.* TABLES ──")
print("─" * 75)
print(f"{'Table':<40} {'Contains':<35}")
print("─" * 75)
print(f"{'system.lakeflow.jobs':<40} {'Job metadata: name, owner, task count, schedule':<35}")
print(f"{'system.lakeflow.job_runs':<40} {'Per-run execution: start/end, status, task results':<35}")
print(f"{'system.lakeflow.job_tasks':<40} {'Per-task execution: task type, duration, status':<35}")
print(f"{'system.lakeflow.pipelines':<40} {'Pipeline metadata: name, owner, schedule':<35}")
print(f"{'system.lakeflow.pipeline_updates':<40} {'Per-update: state, start/end, cause':<35}")
print(f"{'system.lakeflow.pipeline_events':<40} {'Per-event: progress, data quality, errors':<35}")
print("─" * 75)

print("\n── FRESHNESS MONITOR: pipelines where last update is older than SLA ──")
freshness_sql = """
WITH last_success AS (
  SELECT pipeline_id, pipeline_name,
         MAX(CASE WHEN state = 'COMPLETED' THEN update_end_ts END) AS last_success_ts
  FROM   system.lakeflow.pipeline_updates
  WHERE  update_start_ts >= DATEADD(day, -30, CURRENT_DATE())
  GROUP BY 1, 2
)
SELECT pipeline_name,
       last_success_ts,
       DATEDIFF(minute, last_success_ts, CURRENT_TIMESTAMP()) AS minutes_since_success
FROM   last_success
WHERE  last_success_ts IS NOT NULL
  AND  DATEDIFF(minute, last_success_ts, CURRENT_TIMESTAMP()) > 120  -- SLA: 2 hours
ORDER BY minutes_since_success DESC;
"""
print(freshness_sql)

print("\n── FAILURE-RATE MONITOR: jobs below 95% success in last 7 days ──")
failure_sql = """
WITH job_stats AS (
  SELECT job_id, job_name,
         COUNT(*) AS total_runs,
         SUM(CASE WHEN result_state = 'SUCCEEDED' THEN 1 ELSE 0 END) AS success_runs
  FROM   system.lakeflow.job_runs
  WHERE  start_time >= DATEADD(day, -7, CURRENT_DATE())
  GROUP BY 1, 2
)
SELECT job_name, total_runs, success_runs,
       ROUND(success_runs * 100.0 / total_runs, 1) AS success_pct
FROM   job_stats
WHERE  total_runs >= 5  -- only jobs with enough history
  AND  success_runs * 100.0 / total_runs < 95.0
ORDER BY success_pct ASC;
"""
print(failure_sql)

print("\n── LAKEHOUSE MONITORING (native feature) vs AD-HOC QUERIES ──")
print("  Lakehouse Monitoring (Databricks native):")
print("    Attach a monitor to ANY UC table. Databricks auto-tracks:")
print("    - DRIFT: schema drift, data type changes, new/missing columns")
print("    - FRESHNESS: last write timestamp, staleness vs expected cadence")
print("    - VOLUME: row count trend, anomaly detection (sudden drops/spikes)")
print("    - CUSTOM SQL METRICS: your own metric SQL (e.g., null_rate, unique_count)")
print("    Output: a time-series metrics table + a drift snapshot table in UC.")
print("  Ad-hoc system-table queries (what we wrote above):")
print("    One-off analysis, flexible, but you build and maintain the queries.")
print("  Use both: Lakehouse Monitoring for the persistent baseline; ad-hoc for investigation.")

print("\n── METRICS-TABLE PATTERN ──")
print("  1. At the END of each pipeline run, write metrics to a Delta table:")
print("     INSERT INTO platform.pipeline_metrics")
print("     VALUES (pipeline_name, run_id, NOW(), row_count, dq_pass_rate, duration_sec)")
print("  2. Create a SQL alert on the metrics table:")
print("     - Freshness: WHERE minutes_since_last_run > SLA")
print("     - Volume: WHERE row_count < expected_min")
print("     - DQ: WHERE dq_pass_rate < 0.95")
print("  3. Alert triggers → Slack/PagerDuty → on-call engineer investigates.")
print("  4. Dashboard: read the metrics table to show trends over time.")
print()
print("  This pattern is the backbone of platform observability:")
print("    pipeline writes metrics → alert reads metrics → on-call acts → dashboard shows trends.")
print("\nInterview one-liner: 'system.lakeflow.* gives you the WHAT (which pipeline is stale);")
print("  Lakehouse Monitoring gives you the WHY (drift, volume anomalies); the metrics-table")
print("  pattern wires both into alerts that wake a human.'")

## 🏁 Day 16 Score Card

Track your performance — be honest, the point is to find gaps *before* the real interview.

| Q | Topic | Tag | Attempted (✓/✗) | Confident (1-5) | Revisit? |
| --- | --- | --- | --- | --- | --- |
| Q1 | Cohort retention grid from activity events | ADVANCED | | | |
| Q2 | Sessionization via gaps-and-islands | HARD | | | |
| Q3 | LRU cache (doubly-linked list + dict) | ADVANCED | | | |
| Q4 | Rate limiter (token bucket + sliding window) | ADVANCED | | | |
| Q5 | Handling skewed data with salting | ADVANCED | | | |
| Q6 | SCD Type 2 MERGE INTO with Delta Lake | ADVANCED | | | |
| Q7 | AQE internals (coalesce, join switch, skew split) | ADVANCED | | | |
| Q8 | Speculative execution: straggler detection | ADVANCED | | | |
| Q9 | Watermarks & late data in Structured Streaming | ADVANCED | | | |
| Q10 | Exactly-once semantics end-to-end | HARD | | | |
| Q11 | Circuit breaker pattern | HARD | | | |
| Q12 | Conformed dimensions & drill-across queries | ADVANCED | | | |
| Q13 | Late-arriving facts vs dimensions | ADVANCED | | | |
| Q14 | Snowflake Search Optimization Service (SOS) | ADVANCED | | | |
| Q15 | Row access policies & dynamic data masking | ADVANCED | | | |
| Q16 | Jobs orchestration: task types, deps, triggers | ADVANCED | | | |
| Q17 | 3 AM failure debugging walkthrough | HARD | | | |
| Q18 | Lakehouse Monitoring + system.lakeflow.* | ADVANCED | | | |

**Self-review prompts:**

1. Which question took the longest to articulate out loud? Practice it twice more tomorrow.
2. For every ✗: write the one-sentence core idea on a flashcard (e.g. *"gaps-and-islands: lag → gap → flag → cumulative sum → group"*).
3. Day 16 deepens Week 3: you should now be comfortable defending trade-offs for sessionization (window vs recursion), salting vs AQE vs broadcast, and exactly-once's four pillars. If any of these are shaky, revisit Day 15's caching and bitemporal content — they build on the same foundations.
4. Tomorrow: Day 17 (ADVANCED — pushing further into HARD territory with GenAI data pipelines and streaming patterns).